# 01 - Clean EMS incident dispatch data
**Input:** `data/raw/ems_incidents_<year>.csv.gz` (NYC Open Data `76xm-jjuj`, fetched by `scripts/fetch_data.py ems`)
**Output:** `data/interim/ems_incidents_clean.csv.gz`

One row per incident. We keep what the forecaster and the simulator need: when/where demand arose, acuity, and
the empirical durations of each stage of an ambulance job (used as simulator service times).

In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = next(p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents] if (p / "README.md").exists())
sys.path.insert(0, str(ROOT / "src"))
RAW, INTERIM, PROCESSED = ROOT / "data/raw", ROOT / "data/interim", ROOT / "data/processed"
NB_DATA = ROOT / "notebooks/data"
INTERIM.mkdir(parents=True, exist_ok=True); PROCESSED.mkdir(parents=True, exist_ok=True)
pd.set_option("display.width", 160); pd.set_option("display.max_columns", 40)

## Load and type

In [ ]:
files = sorted(RAW.glob("ems_incidents_*.csv.gz"))
assert files, "Run: python scripts/fetch_data.py ems"
print([f.name for f in files])

DT = ["incident_datetime", "first_assignment_datetime", "first_activation_datetime", "first_on_scene_datetime",
      "first_to_hosp_datetime", "first_hosp_arrival_datetime", "incident_close_datetime"]
NUM = ["final_severity_level_code", "dispatch_response_seconds_qy",
       "incident_response_seconds_qy", "incident_travel_tm_seconds_qy"]

DROP = ["initial_severity_level_code", "held_indicator", "citycouncildistrict", "valid_dispatch_rspns_time_indc", "first_assignment_datetime"]
CAT = ["initial_call_type", "final_call_type", "borough", "zipcode", "policeprecinct", "incident_dispatch_area",
       "special_event_indicator", "standby_indicator", "transfer_indicator", "valid_incident_rspns_time_indc"]

def load(f):
    df = pd.read_csv(f, dtype=str).drop(columns=DROP)      # drop unused columns early: 4.5M rows is memory-heavy
    for c in DT:
        if c in df: df[c] = pd.to_datetime(df[c], errors="coerce")
    for c in NUM:
        if c in df: df[c] = pd.to_numeric(df[c], errors="coerce")
    for c in CAT: df[c] = df[c].astype("category")
    return df

raw = pd.concat([load(f) for f in files], ignore_index=True)   # categories unify to object only if they differ; fine
print(f"{len(raw):,} rows"); raw.head(3)

## Profile: missingness and obvious problems

In [ ]:
print((raw.isna().mean() * 100).round(1).sort_values(ascending=False).head(15).to_string())
print("\nborough:", raw.borough.value_counts(dropna=False).to_dict())
print("\nzipcode sample of non-5-digit values:", raw.loc[~raw.zipcode.astype(str).str.fullmatch(r"\d{5}"), "zipcode"].value_counts(dropna=False).head(8).to_dict())
print("\nduplicate incident_id:", raw.incident_id.duplicated().sum())

## Cleaning rules (each count is reported so we can defend it in the pitch)
1. Drop duplicate `incident_id` and rows with no `incident_datetime`.
2. **Demand set** keeps every real call (needed for forecasting). Drop `standby` and `transfer` incidents (they are not new demand).
3. Standardise `borough` (upper-case; `RICHMOND / STATEN ISLAND` -> `STATEN ISLAND`) and `zipcode` (5 digits, else missing).
4. **Response set**: `incident_response_seconds` only trusted where `valid_incident_rspns_time_indc == 'Y'` and 0 < t <= 3h.
5. Also drop `STNDBY` call types (units parked at events; flagged inconsistently).
6. Derive stage durations from timestamps; discard negative or absurd (> 6h) values by setting them to NaN (the row stays).

In [ ]:
n0 = len(raw)
df = raw.drop_duplicates("incident_id").dropna(subset=["incident_datetime"]).copy()
print(f"dedupe/no-timestamp: -{n0 - len(df):,}")

n1 = len(df)
df = df[(df.standby_indicator != "Y") & (df.transfer_indicator != "Y") & (df.final_call_type != "STNDBY")]
print(f"standby/transfer removed: -{n1 - len(df):,}")

df["borough"] = df.borough.astype(str).str.upper().str.strip().replace({"RICHMOND / STATEN ISLAND": "STATEN ISLAND", "NAN": np.nan})
df["zipcode"] = df.zipcode.astype(object).where(df.zipcode.astype(str).str.fullmatch(r"\d{5}"))
df["communitydistrict"] = pd.to_numeric(df.communitydistrict, errors="coerce").astype("Int64")

# response time (what the public stats report): call -> first unit on scene
ok = (df.valid_incident_rspns_time_indc == "Y") & df.incident_response_seconds_qy.between(1, 3 * 3600)
df["response_s"] = df.incident_response_seconds_qy.where(ok)
df["dispatch_s"] = df.dispatch_response_seconds_qy.where(df.dispatch_response_seconds_qy.between(0, 3 * 3600))
df["travel_s"] = df.incident_travel_tm_seconds_qy.where(df.incident_travel_tm_seconds_qy.between(1, 3 * 3600))

def dur(a, b, cap=6 * 3600):
    d = (df[b] - df[a]).dt.total_seconds()
    return d.where(d.between(0, cap))

df["onscene_s"] = dur("first_on_scene_datetime", "first_to_hosp_datetime")        # patient care until departing for hospital
df["to_hosp_s"] = dur("first_to_hosp_datetime", "first_hosp_arrival_datetime")     # transport leg
df["job_s"] = dur("first_activation_datetime", "incident_close_datetime")          # whole unit-busy period (proxy)
df["transported"] = df.first_to_hosp_datetime.notna()

# Acuity (ASSUMPTION: code 1 = most critical ... 8 = least; confirm against NYC data dictionary)
df["acuity"] = pd.cut(df.final_severity_level_code, [0, 3, 6, 9], labels=["critical(1-3)", "mid(4-6)", "low(7-8)"])
print(f"\nclean rows: {len(df):,}")
df[["response_s", "dispatch_s", "travel_s", "onscene_s", "to_hosp_s", "job_s"]].describe(percentiles=[.5, .9, .99]).round(0)

## Sanity checks and first look at the data

In [ ]:
df["year"] = df.incident_datetime.dt.year
print(df.groupby("year").size().to_string())

fig, ax = plt.subplots(1, 3, figsize=(16, 4))
df.set_index("incident_datetime").resample("W").size().plot(ax=ax[0], title="Incidents per week")
heat = df.groupby([df.incident_datetime.dt.dayofweek, df.incident_datetime.dt.hour]).size().unstack()
im = ax[1].imshow(heat, aspect="auto"); ax[1].set_title("Incidents by weekday x hour"); ax[1].set_xlabel("hour"); ax[1].set_ylabel("Mon=0")
(df.groupby("borough").response_s.agg(["mean", lambda s: s.quantile(.9)]) / 60).plot.bar(ax=ax[2], title="Response minutes by borough (mean, P90)")
plt.tight_layout(); plt.show()

print("% responded within 8 min:", (df.response_s <= 480).mean().round(3), "| within 10 min:", (df.response_s <= 600).mean().round(3))
print(df.groupby("acuity", observed=True).response_s.agg(["count", "median"]).round(0))

## Save
Service-time columns (`onscene_s`, `to_hosp_s`, `job_s`, `travel_s`) feed the simulator; `acuity` lets us restrict to low/mid-acuity calls.

In [ ]:
KEEP = ["incident_id", "incident_datetime", "initial_call_type", "final_call_type", "final_severity_level_code", "acuity",
        "borough", "zipcode", "communitydistrict", "policeprecinct", "incident_dispatch_area", "special_event_indicator",
        "dispatch_s", "travel_s", "response_s", "onscene_s", "to_hosp_s", "job_s", "transported"]
df[KEEP].to_csv(INTERIM / "ems_incidents_clean.csv.gz", index=False)
print("saved", len(df), "rows")